# Choosing `lam_et`

Two jobs, in order:

**Diagnose first (sections 1-4).** "The reconstruction just looks like T1" has a precise
meaning in residual mode: the predicted residual has collapsed toward zero, so
`T1ce_pred = pred + T1` reduces to `T1`. That is a *degenerate optimum*, not a bug -- T1
already carries all the bulk anatomy, so copying it scores well on PSNR and SSIM while
predicting no enhancement whatsoever. Global metrics cannot see it. `resid_ratio` can:

    resid_ratio = rms(predicted residual) / rms(true residual)
    1.0 -> right magnitude      0.0 -> output is exactly the T1 anchor

Before sweeping, find out *why*. A collapse driven by LR backtracking or a misaligned ET
mask will not be fixed by any value of `lam_et`, and sweeping would just burn GPU hours
confirming that every value is bad.

**Then sweep (sections 5-7).** Short fine-tunes from one checkpoint, one per `lam_et`.
Far cheaper than training arms from scratch, and enough to rank the values.

In [ ]:
import os, json, copy, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

os.chdir('/scratch/ee2178/ImMAP')          # <-- EDIT to your repo root

# The run to diagnose and sweep from. Use a checkpoint that has ALREADY trained a while --
# the sweep fine-tunes from it, so a good starting point makes the comparison meaningful.
REF_CFG    = "trained_nets/brats/Residual_Unet_T1ce_ET/config.json"
SPLIT      = "val"
CROP       = 192
BATCH      = 8
SEED       = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from models import build_model
from training.common import (load_model, load_ckpt, apply_loss_mask,
                             region_loss, region_psnr)
from training.metrics import compute_metrics
from training.losses import LOSS_REGISTRY
import datasets
from datasets.registry import build_loader

assert os.path.exists(REF_CFG), f"no such config: {REF_CFG}"
cfg = json.load(open(REF_CFG))
T, D = cfg["training"], cfg["data"]["train"]
RESID   = bool(T.get("residual_mode", False))
SRC_IDX = int(T.get("residual_src_idx", 0))
SCALES  = D.get("scales")
LOSS    = LOSS_REGISTRY[T.get("loss_type", "magnitude-l1")]
print(f"{cfg['experiment']['name']}")
print(f"  model={cfg['model']['type']}  residual={RESID} (src_idx={SRC_IDX})  "
      f"lam_et={T.get('lam_et', 0.0)}  scales={SCALES}")
print(f"  loss={T.get('loss_type')}  lr={cfg['optimizer']['params']['lr']}  "
      f"clip_grad={T.get('clip_grad')}  backtrack_thresh={T.get('backtrack_thresh')}")
if not RESID:
    print("  NOTE: this run is not residual_mode; resid_ratio below is measured against the "
          "T1 anchor anyway, which still detects 'output == T1'.")

## 1. Did the learning rate collapse?

Check this **before** anything else. The training loop backtracks whenever
`avg_loss > best_loss + backtrack_thresh`, restoring the last checkpoint and multiplying the
LR by `backtrack_factor`. The ET term is high-variance by construction -- enhancing tumor is
well under 1% of a slice and absent from many of them, so batch-to-batch loss swings grow
when `lam_et` is large. Enough spurious backtracks and the LR decays to nothing, training
stalls near initialization, and a residual net near initialization outputs ~0. That produces
exactly the "looks like T1" symptom, and **no value of `lam_et` fixes it** -- the fix is a
larger `backtrack_thresh` (it is an absolute loss delta, and the ET term raised the scale).

The optimizer state in the checkpoint carries the LR that was actually in use.

In [ ]:
ck = torch.load(cfg["paths"]["ckpt"], map_location="cpu", weights_only=False)
lrs = [g["lr"] for g in ck["optimizer_state_dict"]["param_groups"]]
lr0 = float(cfg["optimizer"]["params"]["lr"])
step = ck.get("step", 0)
spe = int(T["steps_per_epoch"])
print(f"checkpoint step {step}  (~epoch {step // max(spe,1)} of {T['num_epochs']})")
print(f"initial lr {lr0:.3e}   ->   checkpoint lr {lrs[0]:.3e}   ratio {lrs[0]/lr0:.4f}")

sched = cfg["scheduler"]
if sched["type"] == "CosineAnnealingLR":
    tmax, emin = sched["params"]["T_max"], sched["params"]["eta_min"]
    expect = emin + 0.5 * (lr0 - emin) * (1 + math.cos(math.pi * min(step, tmax) / tmax))
    print(f"cosine schedule alone would give {expect:.3e} at this step")
    n_bt = math.log(max(lrs[0], 1e-12) / max(expect, 1e-12)) / math.log(T["backtrack_factor"])
    print(f"-> implies ~{max(n_bt, 0):.0f} backtracking events beyond the schedule")
    if lrs[0] < 0.2 * expect:
        print("\n*** LR IS FAR BELOW SCHEDULE: backtracking has been firing. Training has")
        print("    effectively stalled. Raise backtrack_thresh (it is an ABSOLUTE loss delta,")
        print("    and the ET term increased the loss scale AND its variance) before reading")
        print("    anything into lam_et. ***")
    else:
        print("\nLR is close to schedule -- backtracking is not the cause; continue to section 2.")

## 2. Has the residual collapsed?

The direct measurement. Also reports the ET-restricted ratio -- a model can keep global
residual energy (edges) while giving up on the tumor specifically, and vice versa.

In [ ]:
base = dict(cfg["data"][SPLIT])
base.update(name="synthesis", scales=None, et_mask=True, center_crop=CROP,
            random_flips=False, num_workers=0, batch_size=BATCH)
base.pop("crop_size", None)
torch.manual_seed(SEED)
loader = build_loader(base, shuffle=True, drop_last=False)

def to_scaled(X_raw):
    """raw z-scored -> the run's training space; returns (Xs, s_y)."""
    inp, tgt = list(D["input_idx"]), int(D["target_idx"][0])
    if SCALES is None:
        return X_raw, 1.0
    s_in = torch.tensor([float(SCALES[i]) for i in inp], device=X_raw.device).view(1, -1, 1, 1)
    return X_raw / s_in, float(SCALES[tgt])

@torch.no_grad()
def forward_resid(net, X_raw, y_raw):
    """-> (pred_resid, gt_resid, t1ce_pred), all in the run's SCALED space."""
    Xs, s_y = to_scaled(X_raw)
    ys = y_raw / s_y
    src = Xs[:, SRC_IDX:SRC_IDX + 1]
    out = net(Xs)
    p = out[0] if isinstance(out, (tuple, list)) else out
    pred_resid = p if RESID else (p - src)
    return pred_resid, ys - src, (pred_resid + src)

net = load_model(REF_CFG, device=device); net.eval()
X, y, mask, et = [t.to(device) for t in next(iter(loader))]

pr, gr, t1ce = forward_resid(net, X, y)
pr_m, gr_m = pr * mask, gr * mask
rms = lambda t: float(t.pow(2).mean().sqrt())
ratio = rms(pr_m) / max(rms(gr_m), 1e-8)
em = et.bool().expand_as(pr)
et_ratio = (float(pr_m[em].pow(2).mean().sqrt()) / max(float(gr_m[em].pow(2).mean().sqrt()), 1e-8)
            if em.any() else float("nan"))

print(f"rms(GT residual)   = {rms(gr_m):.4f}")
print(f"rms(pred residual) = {rms(pr_m):.4f}")
print(f"resid_ratio        = {ratio:.4f}      <- 1.0 good, 0.0 = output IS the T1 anchor")
print(f"et_resid_ratio     = {et_ratio:.4f}      <- inside the tumor only")
if ratio < 0.25:
    print("\n*** COLLAPSED: the network is predicting almost no residual, so the output is")
    print("    essentially T1 with noise. Sections 1 and 3 say whether this is a stalled LR")
    print("    or a mask problem; if neither, lam_et is too high and section 5 will show it. ***")

## 3. Is the ET mask aligned with the images?

A mask that points at the wrong pixels supervises the wrong pixels, and at a 300x per-pixel
weight that is enough to wreck training. The backfill reconstructed alignment from the h5's
`crop_size` attr and `slice_index`, which is only checked for files it actually wrote -- if
your `et` came from the original generation pass, this has never been verified.

The test needs no ground truth: enhancing tumor is *bright in T1ce relative to T1*, so the
true residual inside the ET mask should be strongly positive. If the mask is misaligned it
will sit on tissue with a near-zero mean residual, indistinguishable from background.

In [ ]:
vals_in = gr_m[em] if em.any() else torch.zeros(0, device=device)
brain = mask.bool().expand_as(gr_m) & ~em
vals_out = gr_m[brain]

if vals_in.numel() == 0:
    print("no ET in this batch -- try another SEED")
else:
    mi, mo = float(vals_in.mean()), float(vals_out.mean())
    pi = float((vals_in > 0).float().mean())
    print(f"mean TRUE residual inside ET  = {mi:+.4f}   ({pi:.1%} of ET pixels positive)")
    print(f"mean TRUE residual outside ET = {mo:+.4f}")
    print(f"separation = {mi - mo:+.4f}")
    if mi > 0 and pi > 0.65 and (mi - mo) > 0.05:
        print("\n-> ET mask looks correctly aligned (tumor is enhancing, as it should be).")
    else:
        print("\n*** ET MASK LOOKS WRONG: pixels inside the mask are not systematically")
        print("    brighter in T1ce than T1. Check alignment before touching lam_et --")
        print("    re-run preprocessing/cmap.py --add-seg-only --overwrite-seg, or verify")
        print("    et_labels matches this BraTS release. ***")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].hist(vals_out.cpu().numpy(), bins=120, alpha=.6, density=True, label="brain, non-ET")
    ax[0].hist(vals_in.cpu().numpy(), bins=120, alpha=.6, density=True, label="inside ET")
    ax[0].axvline(0, color="k", lw=.8); ax[0].legend(fontsize=8)
    ax[0].set_title("true residual T1ce - T1", fontsize=9); ax[0].set_xlabel("intensity")
    i = int(et.flatten(1).sum(1).argmax())
    v = float(gr_m[i].abs().quantile(0.995))
    ax[1].imshow(gr_m[i, 0].cpu().numpy(), cmap="bwr", vmin=-v, vmax=v)
    ax[1].contour(et[i, 0].cpu().numpy(), levels=[.5], colors="k", linewidths=.8)
    ax[1].set_title("GT residual with ET outline: the outline should sit on the red blob",
                    fontsize=9); ax[1].set_xticks([]); ax[1].set_yticks([])
    plt.tight_layout(); plt.show()

## 4. What `lam_et` actually buys, on *your* data

The ET term is area-normalized, so the two loss terms have different denominators:

```
loss = (1/N_all) * sum_brain |e|   +   lam_et * (1/N_ET) * sum_ET |e|
```

which makes a single ET pixel worth `1 + lam_et * N_all/N_ET` ordinary pixels. With ET well
under 1% of a slice that multiplier is in the hundreds. Measured on real batches below --
substitute these for any assumed numbers.

In [ ]:
N_M = 20
bf, ef = [], []
it = iter(build_loader(base, shuffle=True, drop_last=False))
for _ in range(N_M):
    try: _, _, bm, be = next(it)
    except StopIteration: break
    bf.append(float(bm.mean())); ef.append(float(be.mean()))
brain_frac, et_frac = float(np.mean(bf)), float(np.mean(ef))
print(f"measured over {len(bf)} batches:  brain_frac={brain_frac:.4f}   et_frac={et_frac:.5f}")
print(f"ET is {et_frac*100:.3f}% of the FOV, {et_frac/brain_frac*100:.3f}% of the brain\n")

print(f"{'lam_et':>8} | {'ET px weight':>13} | {'ET share of loss':>17}")
print("-" * 46)
GRID = [0.0, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
for l in GRID:
    w = 1 + l / max(et_frac, 1e-9)
    share = l / (brain_frac + l) if (brain_frac + l) > 0 else 0.0
    print(f"{l:8g} | {w:12.0f}x | {share:16.1%}")
print("\n(share assumes uniform error; the real share is higher, since error concentrates in ET)")
cur = float(T.get("lam_et", 0.0))
print(f"\nthis run used lam_et={cur:g} -> ET pixel weight ~{1 + cur/max(et_frac,1e-9):.0f}x, "
      f"~{cur/(brain_frac+cur) if brain_frac+cur>0 else 0:.0%} of the objective")

## 5. The sweep

Fine-tunes an independent **copy** of the reference checkpoint for `SWEEP_STEPS` at each
`lam_et`, on the same batches in the same order, then evaluates all arms on the same held-out
batches. Same starting weights + same data order means differences are attributable to
`lam_et` alone.

This measures *which weight is best from here*, not what training from scratch would give --
enough to rank values and spot where collapse begins, at a tiny fraction of the cost.

Optimizer state is **not** restored: it carries the decayed LR from any backtracking, which
would silently make every arm train at a crawl. `SWEEP_LR` sets it explicitly.

In [ ]:
SWEEP_GRID  = [0.0, 0.05, 0.15, 0.5, 1.0]
SWEEP_STEPS = 300
SWEEP_LR    = 1e-4          # explicit: do NOT inherit a backtracked LR from the checkpoint
EVAL_BATCHES = 6

train_cfg = dict(cfg["data"]["train"])
train_cfg.update(name="synthesis", scales=None, et_mask=True, num_workers=0,
                 batch_size=cfg["data"]["train"].get("batch_size", 8))
use_mask = bool(T.get("use_mask", True))
clip = T.get("clip_grad")

def fresh_net():
    n = build_model(cfg).to(device)
    n.load_state_dict(torch.load(cfg["paths"]["ckpt"], map_location=device,
                                 weights_only=False)["model_state_dict"])
    return n

def batches(n, seed):
    torch.manual_seed(seed)
    ld = build_loader(train_cfg, shuffle=True, drop_last=True)
    out, it = [], iter(ld)
    for _ in range(n):
        try: b = next(it)
        except StopIteration:
            it = iter(ld); b = next(it)
        out.append([t.to(device) for t in b])
    return out

train_batches = batches(SWEEP_STEPS, SEED + 10)      # identical stream for every arm
eval_batches  = batches(EVAL_BATCHES, SEED + 99)
print(f"{len(train_batches)} train steps, {len(eval_batches)} eval batches, "
      f"grid {SWEEP_GRID}")

@torch.no_grad()
def evaluate(n):
    n.eval(); acc = []
    for bX, by, bm, bet in eval_batches:
        pr, gr, _ = forward_resid(n, bX, by)
        prm, grm = pr * bm, gr * bm
        _, s_y = to_scaled(bX)
        src = to_scaled(bX)[0][:, SRC_IDX:SRC_IDX + 1] * bm
        m = compute_metrics(grm + src, prm + src)      # T1ce domain, brain-masked
        e = float(region_psnr(grm, prm, bet))
        r = float(prm.pow(2).mean().sqrt()) / max(float(grm.pow(2).mean().sqrt()), 1e-8)
        acc.append((float(m["psnr"]), float(m["ssim"]), e, r))
    return np.array(acc).mean(0)

results = {}
for lam in SWEEP_GRID:
    n = fresh_net(); n.train()
    opt = torch.optim.Adam(n.parameters(), lr=SWEEP_LR)
    for bX, by, bm, bet in train_batches:
        Xs, s_y = to_scaled(bX)
        src = Xs[:, SRC_IDX:SRC_IDX + 1]
        tgt = (by / s_y) - src if RESID else (by / s_y)
        out = n(Xs)
        p = out[0] if isinstance(out, (tuple, list)) else out
        tm, pm = apply_loss_mask(tgt, p, bm, use_mask)
        loss = LOSS(pm, tm, None)
        if lam:
            loss = loss + lam * region_loss(LOSS, pm, tm, bet)
        opt.zero_grad(); loss.backward()
        if clip is not None:
            nn.utils.clip_grad_norm_(n.parameters(), clip)
        opt.step()
        if hasattr(n, "project"):
            n.project()
    results[lam] = evaluate(n)
    p_, s_, e_, r_ = results[lam]
    print(f"lam_et={lam:<6g} PSNR={p_:7.3f}  SSIM={s_:.4f}  ET_PSNR={e_:7.3f}  "
          f"resid_ratio={r_:.3f}")
    del n, opt
    if device.type == "cuda":
        torch.cuda.empty_cache()

## 6. Read the sweep

Three curves against `lam_et`. The one that matters most here is `resid_ratio`: watch where
it starts falling away from ~1, because that is where the ET weight begins pushing the model
toward copying T1 rather than predicting enhancement.

In [ ]:
lams = sorted(results)
P = np.array([results[l] for l in lams])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
x = np.arange(len(lams))

ax[0].plot(x, P[:, 0], "o-", label="global PSNR")
ax[0].set_ylabel("PSNR (dB)"); ax[0].set_title("global fidelity", fontsize=10)
ax[1].plot(x, P[:, 2], "o-", color="tab:red")
ax[1].set_ylabel("ET PSNR (dB)"); ax[1].set_title("tumor fidelity", fontsize=10)
ax[2].plot(x, P[:, 3], "o-", color="tab:green")
ax[2].axhline(1.0, ls="--", c="0.6", lw=1)
ax[2].axhline(0.25, ls=":", c="tab:red", lw=1)
ax[2].set_ylabel("rms(pred) / rms(gt)"); ax[2].set_title("resid_ratio: collapse detector",
                                                         fontsize=10)
for a in ax:
    a.set_xticks(x); a.set_xticklabels([f"{l:g}" for l in lams])
    a.set_xlabel("lam_et"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

best_et = lams[int(np.argmax(P[:, 2]))]
safe = [l for l, r in zip(lams, P[:, 3]) if r > 0.5]
print(f"highest ET PSNR at lam_et={best_et:g}")
print(f"resid_ratio stays above 0.5 for lam_et in {safe if safe else '(none)'}")
p0 = P[lams.index(0.0), 0] if 0.0 in lams else P[0, 0]
ok_ = [l for l, p_ in zip(lams, P[:, 0]) if p_ >= p0 - 0.5]
print(f"global PSNR within 0.5 dB of the lam_et=0 arm for {ok_}")
cand = [l for l in lams if l in set(safe) & set(ok_) and l > 0]
print(f"\n-> candidates satisfying both guards: {cand if cand else '(none -- see below)'}")
if not cand:
    print("   No positive lam_et passes. Either the grid is too coarse at the low end")
    print("   (try 0.01-0.05), or ET weighting is not helping this model and the honest")
    print("   answer is lam_et=0.")

## 7. Look at the images

Numbers can agree while the images disagree. Residual maps for each swept `lam_et`, shared
diverging scale, ET outlined -- a collapsed arm is unmistakable here (nearly all white).

In [ ]:
bX, by, bm, bet = eval_batches[0]
i = int(bet.flatten(1).sum(1).argmax())
panels = []
for lam in lams:
    n = fresh_net(); n.train()
    opt = torch.optim.Adam(n.parameters(), lr=SWEEP_LR)
    for tb in train_batches:
        tX, ty, tm_, tet = tb
        Xs, s_y = to_scaled(tX)
        src = Xs[:, SRC_IDX:SRC_IDX + 1]
        tgt = (ty / s_y) - src if RESID else (ty / s_y)
        out = n(Xs); p = out[0] if isinstance(out, (tuple, list)) else out
        tt_, pp_ = apply_loss_mask(tgt, p, tm_, use_mask)
        loss = LOSS(pp_, tt_, None)
        if lam:
            loss = loss + lam * region_loss(LOSS, pp_, tt_, tet)
        opt.zero_grad(); loss.backward()
        if clip is not None: nn.utils.clip_grad_norm_(n.parameters(), clip)
        opt.step()
        if hasattr(n, "project"): n.project()
    with torch.no_grad():
        pr_, gr_, _ = forward_resid(n, bX, by)
    panels.append((lam, (pr_ * bm)[i, 0].cpu().numpy()))
    del n, opt
    if device.type == "cuda": torch.cuda.empty_cache()

gt_panel = (gr_ * bm)[i, 0].cpu().numpy()
v = float(np.quantile(np.abs(gt_panel), 0.995))
fig, ax = plt.subplots(1, len(panels) + 1, figsize=(2.5 * (len(panels) + 1), 3.0))
ax[0].imshow(gt_panel, cmap="bwr", vmin=-v, vmax=v)
ax[0].set_title("GT residual", fontsize=9)
for k, (lam, img) in enumerate(panels, start=1):
    ax[k].imshow(img, cmap="bwr", vmin=-v, vmax=v)
    ax[k].set_title(f"lam_et={lam:g}", fontsize=9)
for a in ax:
    a.contour(bet[i, 0].cpu().numpy(), levels=[.5], colors="k", linewidths=.6)
    a.set_xticks([]); a.set_yticks([])
plt.suptitle("predicted residual, shared scale -- an all-white panel is a collapsed arm",
             y=1.02, fontsize=10)
plt.tight_layout(); plt.show()